In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
print(df.shape)

(7043, 33)


In [4]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [5]:
df.isnull().sum().sum()

np.int64(5174)

In [6]:
leakage_columns = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "Churn Reason"
]

In [7]:
[col for col in leakage_columns if col in df.columns]

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Churn Label',
 'Churn Score',
 'Churn Reason']

In [8]:
df = df.drop(
    columns=leakage_columns,
    errors="ignore"
)


In [9]:
df.shape

(7043, 21)

In [10]:
target = "Churn Value"


In [11]:
df[target].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

In [12]:
X = df.drop(target, axis=1)

y = df[target]

In [13]:
print(X.shape)
print(y.shape)

(7043, 20)
(7043,)


In [14]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns

categorical_cols

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Total Charges'],
      dtype='str')

In [15]:
encoder_dict = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoder_dict[col] = le

In [16]:
X.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV
0,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,157,3239
1,0,0,0,1,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,925,2701
2,0,0,0,1,8,1,2,1,0,0,2,0,2,2,0,1,2,99.65,6104,5372
3,0,0,1,1,28,1,2,1,0,0,2,2,2,2,0,1,2,104.80,2646,5003
4,1,0,0,1,49,1,2,1,0,2,2,0,2,2,0,1,0,103.70,4265,5340


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [18]:
print(X_train.shape)
print(X_test.shape)

(5634, 20)
(1409, 20)


In [19]:
scaler = StandardScaler()

In [20]:
X_train_scaled = scaler.fit_transform(
    X_train
)

In [21]:
X_test_scaled = scaler.transform(
    X_test
)

In [22]:
X_train_scaled.shape

(5634, 20)

In [23]:
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [24]:
processed_df = pd.concat(
    [X, y],
    axis=1
)

processed_df.to_csv(
    "../data/processed/processed_telco.csv",
    index=False
)

In [25]:
processed_df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV,Churn Value
0,1,0,0,0,2,1,0,0,2,2,...,0,0,0,0,1,3,53.85,157,3239,1
1,0,0,0,1,2,1,0,1,0,0,...,0,0,0,0,1,2,70.70,925,2701,1
2,0,0,0,1,8,1,2,1,0,0,...,0,2,2,0,1,2,99.65,6104,5372,1
3,0,0,1,1,28,1,2,1,0,0,...,2,2,2,0,1,2,104.80,2646,5003,1
4,1,0,0,1,49,1,2,1,0,2,...,0,2,2,0,1,0,103.70,4265,5340,1


In [26]:
os.makedirs(
    "../models",
    exist_ok=True
)

In [27]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.joblib"
)

['../models/scaler.joblib']

In [28]:
joblib.dump(
    encoder_dict,
    "../models/encoders.joblib"
)

['../models/encoders.joblib']

In [29]:
os.listdir("../models")

['encoders.joblib', 'scaler.joblib']